# 04 — Regression Modeling (Continuous Prediction)

## What to do
Build regression models to predict continuous health values (BMI, blood glucose, HbA1c level).

## What you'll learn
- Regression vs classification problem setup
- Train regression models
- Evaluate using MSE, RMSE, MAE, R²
- Analyze residuals
- Compare continuous prediction performance

## Step 1: Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

print('Libraries loaded!')

## Step 2: Load and prepare data

**Action:** Load original data and prepare for regression on continuous targets.
**Insight:** We predict continuous health metrics instead of binary diabetes status.

In [ ]:
# Load original data
df = pd.read_csv('data/diabetes_prediction_dataset.csv')

# Select a continuous target (adjust based on your dataset)
continuous_targets = ['bmi', 'blood_glucose_level', 'HbA1c_level']  # Adjust as needed

available_targets = [t for t in continuous_targets if t in df.columns]
print(f'Available continuous targets: {available_targets}')

# Use BMI as example target
target = available_targets[0] if available_targets else None

if target:
    X = df.drop(columns=['diabetes', target])
    y = df[target]
    print(f'\nRegression target: {target}')
    print(f'X shape: {X.shape}')
    print(f'y shape: {y.shape}')
else:
    print('No continuous targets found. Using diabetes as proxy.')

## Step 3: Encode categorical variables

**Action:** Convert categorical features to numeric.
**Insight:** Regression models require numeric input.

In [ ]:
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

if categorical_cols:
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
    print(f'Encoded categorical columns. New shape: {X.shape}')
else:
    print('✓ No categorical columns')

## Step 4: Train-test split

**Action:** Split data into 75% train, 25% test.
**Insight:** Keeps test set completely separate to estimate real-world performance.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print(f'Train set: {X_train.shape}')
print(f'Test set: {X_test.shape}')

## Step 5: Train regression models

**Action:** Train Linear Regression, Ridge, and Random Forest Regressor.
**Insight:** Different models capture different patterns in the data.

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
}

trained_models = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f'  ✓ Complete')

print('\n✓ All regression models trained!')

## Step 6: Make predictions

**Action:** Generate predictions on test set.
**Insight:** Test predictions show real-world performance.

In [ ]:
predictions = {}

for name, model in trained_models.items():
    predictions[name] = model.predict(X_test)

print('✓ Predictions generated')

## Step 7: Evaluate regression models

**Action:** Calculate MSE, RMSE, MAE, and R² scores.
**Insight:** Different metrics reveal different aspects of prediction error.
- MSE/RMSE: Penalizes large errors more
- MAE: Average absolute error
- R²: Proportion of variance explained (0-1, higher is better)

In [ ]:
results_list = []

for name, pred in predictions.items():
    mse = mean_squared_error(y_test, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    
    results_list.append({
        'Model': name,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R² Score': r2
    })

results_df = pd.DataFrame(results_list).sort_values('R² Score', ascending=False)
display(results_df.round(4))

best_model_name = results_df.iloc[0]['Model']
print(f'\n✓ Best model by R² Score: {best_model_name}')

## Step 8: Residual analysis

**Action:** Analyze prediction residuals (errors) for the best model.
**Insight:** Residual patterns reveal model weaknesses. Should be random and centered at 0.

In [ ]:
best_pred = predictions[best_model_name]
residuals = y_test - best_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residual histogram
axes[0].hist(residuals, bins=30, edgecolor='black', color='skyblue')
axes[0].axvline(0, color='red', linestyle='--', label='Zero Error')
axes[0].set_xlabel('Residuals')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Residual Distribution')
axes[0].legend()

# Residual plot
axes[1].scatter(best_pred, residuals, alpha=0.6)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted Values')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot')

plt.tight_layout()
plt.show()

print(f'Mean residual: {residuals.mean():.4f}')
print(f'Std residual: {residuals.std():.4f}')

## Step 9: Actual vs Predicted plot

**Action:** Visualize actual vs predicted values.
**Insight:** Points close to the diagonal line indicate good predictions.

In [ ]:
plt.figure(figsize=(10, 6))

for name, pred in predictions.items():
    plt.scatter(y_test, pred, alpha=0.5, label=name)

# Perfect prediction line
min_val = min(y_test.min(), y_test.max())
max_val = max(y_test.min(), y_test.max())
plt.plot([min_val, max_val], [min_val, max_val], 'k--', label='Perfect', linewidth=2)

plt.xlabel(f'Actual {target}')
plt.ylabel('Predicted')
plt.title('Actual vs Predicted Values')
plt.legend()
plt.grid()
plt.show()

## Step 10: Feature importance (Random Forest)

**Action:** Extract feature importance from Random Forest regressor.
**Insight:** Shows which features most influence continuous predictions.

In [ ]:
rf_model = trained_models['Random Forest']
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance.head(15)['Feature'][::-1], feature_importance.head(15)['Importance'][::-1])
plt.xlabel('Importance')
plt.title(f'Top 15 Features for Predicting {target}')
plt.tight_layout()
plt.show()

## Regression Summary

**What we did:**
1. Loaded continuous target variable
2. Prepared features and encoded categoricals
3. Split train/test
4. Trained 3 regression models
5. Evaluated using MSE, RMSE, MAE, R²
6. Analyzed residuals
7. Compared actual vs predicted
8. Extracted feature importance

**Next:** Continue to clustering for unsupervised pattern discovery.